# Building and pruning decision trees with Palmer Penguins



## Libraries and Data



In [ ]:
import numpy as np
import pandas as pd
from plotnine import *
from plotnine.data import penguins
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.model_selection import (
    GridSearchCV, KFold, StratifiedKFold, train_test_split,
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RANDOM_STATE = 909

# Preserve the supplied notebook's complete-case sample and predictor choices.
df = penguins.copy().reset_index(drop=True).dropna()

regression_features = [
    "flipper_length_mm", "bill_length_mm", "bill_depth_mm"
]
classification_features = [
    "bill_length_mm", "bill_depth_mm"
]

X_regression = df[regression_features]
y_regression = df["body_mass_g"]
X_classification = df[classification_features]
y_classification = df["species"].astype(str)



In [ ]:

def plot_regression_tree(model, feature_names=None):
    """Plot a fitted, single-output DecisionTreeRegressor."""
    tree = model.tree_

    if feature_names is None:
        feature_names = getattr(
            model,
            "feature_names_in_",
            [f"X[{i}]" for i in range(model.n_features_in_)]
        )

    nodes = []
    edges = []
    next_leaf = 0

    def visit(node, depth):
        nonlocal next_leaf

        left = tree.children_left[node]
        right = tree.children_right[node]
        prediction = tree.value[node].item()
        n = tree.n_node_samples[node]

        if left == right:  # Leaf: assign the next horizontal position
            x = next_leaf
            next_leaf += 1
            label = f"Prediction: {prediction:.1f}\nn = {n}"
        else:
            # Position each parent halfway between its children
            x_left = visit(left, depth + 1)
            x_right = visit(right, depth + 1)
            x = (x_left + x_right) / 2

            for child_x in (x_left, x_right):
                edges.append({
                    "x": x, "y": -depth,
                    "xend": child_x, "yend": -(depth + 1)
                })

            feature = feature_names[tree.feature[node]]
            threshold = tree.threshold[node]
            label = f"{feature} ≤ {threshold:.2f}\nn = {n}"

        nodes.append({"x": x, "y": -depth, "label": label})
        return x

    visit(0, 0)

    node_df = pd.DataFrame(nodes)
    edge_df = pd.DataFrame(
        edges, columns=["x", "y", "xend", "yend"]
    )

    return (
        ggplot()
        + geom_segment(
            data=edge_df,
            mapping=aes(x="x", y="y", xend="xend", yend="yend")
        )
        + geom_label(
            data=node_df,
            mapping=aes(x="x", y="y", label="label"),
            size=8,
            fill="aliceblue"
        )
        + labs(title="Regression tree")
        + theme_void()
        + theme(figure_size=(14, 8))
    )


def plot_classification_tree(model, feature_names=None):
    """Plot a fitted, single-output DecisionTreeClassifier."""
    tree = model.tree_

    if feature_names is None:
        feature_names = getattr(
            model,
            "feature_names_in_",
            [f"X[{i}]" for i in range(model.n_features_in_)]
        )

    nodes = []
    edges = []
    next_leaf = 0

    def visit(node, depth):
        nonlocal next_leaf

        left = tree.children_left[node]
        right = tree.children_right[node]

        # Class with the largest count/proportion at this node
        predicted_class = str(
            model.classes_[tree.value[node][0].argmax()]
        )
        n = tree.n_node_samples[node]

        if left == right:  # Terminal node
            x = next_leaf
            next_leaf += 1
            label = f"Class: {predicted_class}\nn = {n}"

        else:  # Split node
            x_left = visit(left, depth + 1)
            x_right = visit(right, depth + 1)
            x = (x_left + x_right) / 2

            for child_x in (x_left, x_right):
                edges.append({
                    "x": x,
                    "y": -depth,
                    "xend": child_x,
                    "yend": -(depth + 1)
                })

            feature = feature_names[tree.feature[node]]
            threshold = tree.threshold[node]

            label = (
                f"{feature} ≤ {threshold:.2f}\n"
                f"Class: {predicted_class}\nn = {n}"
            )

        nodes.append({
            "x": x,
            "y": -depth,
            "label": label,
            "predicted_class": predicted_class
        })
        return x

    visit(0, 0)

    node_df = pd.DataFrame(nodes)
    edge_df = pd.DataFrame(
        edges, columns=["x", "y", "xend", "yend"]
    )

    return (
        ggplot()
        + geom_segment(
            data=edge_df,
            mapping=aes(x="x", y="y", xend="xend", yend="yend")
        )
        + geom_label(
            data=node_df,
            mapping=aes(
                x="x",
                y="y",
                label="label",
                fill="predicted_class"
            ),
            size=8,
            show_legend=False
        )
        + labs(
            title="Classification tree",
            subtitle="Left branch: condition true; right branch: condition false"
        )
        + theme_void()
        + theme(figure_size=(16, 9))
    )

## Train Test Split


In [ ]:
X_train_mass, X_test_mass, y_train_mass, y_test_mass = train_test_split(
    X_regression, y_regression,
    test_size=0.20,
    random_state=909
)

X_train_species, X_test_species, y_train_species, y_test_species = train_test_split(
    X_classification, y_classification,
    test_size=0.20,
    random_state=909,
    stratify=y_classification,
)


## Cross-Validation

In [ ]:
reg_cv = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cls_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

## Regression tree: predict body mass


In [ ]:
reg_full = DecisionTreeRegressor(
    criterion="squared_error", splitter="best",
    max_depth=None, min_samples_split=2, min_samples_leaf=1,
    ccp_alpha=0.0, random_state=RANDOM_STATE,
)
reg_full.fit(X_train_mass, y_train_mass)
print(f"Unpruned tree: {reg_full.get_n_leaves()} leaves, depth {reg_full.get_depth()}")

### Plotting Tree

In [ ]:
plot_regression_tree(reg_full, feature_names=X_train_mass.columns)

### Prediction Error

In [ ]:
y_pred = reg_full.predict(X_test_mass)
print("Test MSE:", mean_squared_error(y_test_mass, y_pred))

### Regression Tree Pruning

In [ ]:
path = reg_full.cost_complexity_pruning_path(X_train_mass, y_train_mass) ## Obtains Alphas
alphas = np.unique(path.ccp_alphas)

reg_search = GridSearchCV(
    estimator=clone(reg_full),
    param_grid={"ccp_alpha": alphas},
    scoring="neg_root_mean_squared_error",
    cv=reg_cv, 
    refit=True, # Will not 
    n_jobs=2 # Use 2 cores
)
reg_search.fit(X_train_mass, y_train_mass)


### Best Tree

In [ ]:
reg_model = reg_search.best_estimator_
print(f"Pruned tree: {reg_model.get_n_leaves()} leaves, depth {reg_model.get_depth()}")

### Prediction Error

In [ ]:
y_pred = reg_model.predict(X_test_mass)
print("Test MSE:", mean_squared_error(y_test_mass, y_pred))

### Pruned Tree

In [ ]:
plot_regression_tree(reg_model, feature_names=X_train_mass.columns)

## Classification tree




In [ ]:
cls_full = DecisionTreeClassifier(
    criterion="gini", 
    splitter="best",
    max_depth=None, 
    min_samples_split=2, 
    min_samples_leaf=1,
    ccp_alpha=0.0, # Full Tree
    random_state=RANDOM_STATE,
)
cls_full.fit(X_train_species, y_train_species)
print(f"Unpruned tree: {cls_full.get_n_leaves()} leaves, depth {cls_full.get_depth()}")

In [ ]:
plot_classification_tree(cls_full, feature_names=X_train_species.columns)

### Classification Error

In [ ]:
y_pred = cls_full.predict(X_test_species)
print("Test Error:", accuracy_score(y_test_species, y_pred))

### Pruning Tree

In [ ]:
path = cls_full.cost_complexity_pruning_path(X_train_species, y_train_species) ## Obtains Alphas
alphas = np.unique(path.ccp_alphas)

cls_search = GridSearchCV(
    estimator=clone(cls_full),
    param_grid={"ccp_alpha": alphas},
    scoring="accuracy",
    cv=cls_cv, 
    refit=True, # Will refit final model
    n_jobs=2 # Use 2 cores
)
cls_search.fit(X_train_species, y_train_species)

### Best Tree

In [ ]:
cls_model = cls_search.best_estimator_
print(f"Pruned tree: {cls_model.get_n_leaves()} leaves, depth {cls_model.get_depth()}")

### Prediction Error

In [ ]:
y_pred = cls_model.predict(X_test_species)
print("Test Error:", accuracy_score(y_test_species, y_pred))

### Pruned Tree

In [ ]:
plot_classification_tree(cls_model, feature_names=X_train_species.columns)